# TSM Scorecard

Runs the queries in `tsm_scorecard.sql` and computes the CS scorecard.

- **Cell 1** — raw scorecard output (per-company, YTD through end of last full quarter)
- **Cell 2** — per-deal expansions / renewals (won, non-zero net new ARR)
- **Cell 3** — scorecard aggregations (TBD)

In [2]:
import os
import pandas as pd
import snowflake.connector
from io import StringIO
from snowflake.connector.util_text import split_statements

conn = snowflake.connector.connect(
    connection_name=os.getenv("SNOWFLAKE_CONNECTION_NAME", "port-analytics-prod")
)
print(f"Connected as {conn.user} / role={conn.role} / db={conn.database}")

def run_query(sql: str) -> pd.DataFrame:
    cur = conn.cursor()
    try:
        cur.execute(sql)
        return cur.fetch_pandas_all()
    finally:
        cur.close()

# Use Snowflake's own SQL splitter so semicolons inside `-- ...` comments
# don't truncate statements.
with open("tsm_scorecard.sql") as f:
    _stmts = [s.strip() for s, _ in split_statements(StringIO(f.read()))]
SQL_SCORECARD  = _stmts[0]
SQL_EXPANSIONS = _stmts[1]
print(f"Loaded {len(_stmts)} statements from tsm_scorecard.sql")
print(f"  [0] scorecard  ({len(SQL_SCORECARD):,} chars)")
print(f"  [1] expansions ({len(SQL_EXPANSIONS):,} chars)")

 pip install snowflake-connector-python[secure-local-storage]


Initiating login request with your identity provider. Press CTRL+C to abort and try again...
Going to open: https://sso.jumpcloud.com/saml2/psf?SAMLRequest=nZJBc9owEIX%2Fikc9Y9mmBKoBMg40jTM0GGxC6KWj2AJUZMnRyjj8%2B8omzKSH5NCbRvre6u2%2BHV6%2FFsI5Mg1cyRHyXQ85TGYq53I3Qqv0tjNADhgqcyqUZCN0YoCux0OghShJWJm9XLKXioFxbCEJpH0YoUpLoihwIJIWDIjJSBL%2BnJHA9UiplVGZEuid5HMFBWDaWIcXSQ7c2tsbUxKM67p2666r9A4Hnudh7xu2VIN8ufCvtqcPeB97XxveEhaP37zdcHkewWe2ns8QkLs0jTvxPEmRE16sTpSEqmA6YfrIM7Zazs4GwDqI7x9v5sl3K1mmv8OHcLZJo0niglT1VtADy1RRVsaWdu0Jb1mOhdpx2300HaHywPNUPiWDeHGfzQ%2FHxaraLcLnde8q3fY3d%2BvNmu57Xv%2F0RH%2BEehplyHm8xBs08UYAFYtkE6qxV15w1fEGnaCf%2Bj7xB6QXuL1u9xdypjZULqlplRfnAMr9UxVlJlSVt%2B6a9AJcwhadt4K09fX4v3od4vcl3rbswQ4%2BmsZK8Ozk3CpdUPNxLr7rtzc872xblLCCchHmuWYANh8hVD3RjBq7zEZXDOHx%2Bdd%2F13n8Fw%3D%3D&RelayState=ver%3A3-hint%3A5187783636193286-ETMsDgAAAaBC8bd1ABRBRVMvQ0JDL1BLQ1M1UGFkZGluZwEAABAAEKA5ybQcISR7jOsv9Y22%2FKcAAACgZ5AVeEHI6O2Tb0tUdj0ICgFFYU9LJZA3mFdCiHz4QfXJWnX%2BUOrQ16nsrnYFppgo%2BXHpEgaZJKS8pQ

 pip install snowflake-connector-python[secure-local-storage]


## Cell 1 — Scorecard raw data

Per-company row set covering ownership, tier, ARR, licence lifecycle, engagement (Gong / building signals / actions / seat utilization), renewal indicators, etc. Attribution is by current owner unless noted.

In [9]:
df_scorecard = run_query(SQL_SCORECARD)
print(f"{len(df_scorecard):,} rows × {len(df_scorecard.columns)} columns")
df_scorecard

ProgrammingError: 001003 (42000): SQL compilation error:
syntax error line 10 at position 54 unexpected '<EOF>'.

## Cell 2 — CW Expansions / Renewals per deal

One row per won `Expansion` or `existingbusiness` deal closed between Jan 1 and end of last full quarter, with `DEAL_NET_NEW_ARR != 0`. Owner attribution is SCD-at-deal-close-time.

In [4]:
df_deals = run_query(SQL_EXPANSIONS)
print(f"{len(df_deals):,} deals, total NET_NEW_ARR = ${df_deals['NET_NEW_ARR'].sum():,.2f}")
df_deals

ProgrammingError: 001003 (42000): SQL compilation error:
syntax error line 1 at position 0 unexpected 'truth'.

## Targets

Per-metric goals used to evaluate each CS owner's aggregated scorecard.
- Percentages are expressed as decimals (0.95 = 95%).
- **Time-to-Onboarding** targets are 0 by default (no lag) — set to whatever SLA you want.
- **Gong meetings** is a per-customer-per-quarter target, so the owner's overall target scales with their customer count.
- **CW Expansion** is a composite score: `(Expansions × 40) + (Avg days_before_renewal / 10) + (Net New ARR / 25,000)`.

In [5]:
GONG_MEETINGS_PER_CUSTOMER_PER_QUARTER = 4  # 4 calls / customer / quarter

TARGETS = {
    'NRR':                        1.35,   # 135%
    'GRR':                        0.95,   # 95%
    'LOGO_RETENTION':             0.95,   # 95%
    'STRATEGIC_TIME_TO_ONBOARD':  0,      # days
    'CORE_PLUS_TIME_TO_ONBOARD':  0,      # days
    'CORE_TIME_TO_ONBOARD':       0,      # days
    'SEAT_UTILIZATION':           0.80,   # 80% share of customers meeting their tenure-based utilization threshold
    'AVG_ACTIONS_MOM':            0.75,   # 75% of MoM transitions non-decreased
    'AVG_BUILDING_SIGNALS':       2500,   # average per customer
}

# --- Dynamic targets (per owner, computed from their customer portfolio) ---
def gong_meetings_target(customer_quarters_active) -> float:
    """Owner-level Gong target = 4 calls per customer per quarter, prorated
    by how much of the measurement window each customer was active.

    `customer_quarters_active` is an iterable of per-customer values, each
    representing 'quarters of the measurement window this customer was
    active for' (e.g. a customer active for all 2 quarters YTD contributes
    2.0; one active for only half of Q2 contributes 0.5). Sum × 4 is the
    owner's target call count."""
    return GONG_MEETINGS_PER_CUSTOMER_PER_QUARTER * sum(customer_quarters_active)

def cw_expansion_score(expansion_logos: int, avg_days_before_renewal: float, net_new_arr: float) -> float:
    if not expansion_logos:
        return 0.0
    return (avg_days_before_renewal * net_new_arr) / (expansion_logos * 2_000_000)

# --- Pretty-print --------------------------------------------------------
import pandas as pd

_FMT = {
    'NRR':                        ('Net Revenue Retention',              lambda v: f'{v:.0%}'),
    'GRR':                        ('Gross Revenue Retention',            lambda v: f'{v:.0%}'),
    'LOGO_RETENTION':             ('Logo Retention',                     lambda v: f'{v:.0%}'),
    'STRATEGIC_TIME_TO_ONBOARD':  ('Strategic — Time to Onboarding',    lambda v: f'{v} days'),
    'CORE_PLUS_TIME_TO_ONBOARD':  ('Core+ — Time to Onboarding',        lambda v: f'{v} days'),
    'CORE_TIME_TO_ONBOARD':       ('Core — Time to Onboarding',         lambda v: f'{v} days'),
    'SEAT_UTILIZATION':           ('Seat Utilization (customers meeting threshold)', lambda v: f'{v:.0%}'),
    'AVG_ACTIONS_MOM':            ('Actions MoM Growth',                 lambda v: f'{v:.0%}'),
    'AVG_BUILDING_SIGNALS':       ('Avg Building Signals',               lambda v: f'{v:,} / customer'),
}

_rows = [{'Metric': label, 'Target': fmt(TARGETS[k])} for k, (label, fmt) in _FMT.items()]
_rows.append({
    'Metric': 'Gong Meetings (dynamic)',
    'Target': f'{GONG_MEETINGS_PER_CUSTOMER_PER_QUARTER} calls / customer / quarter — owner target = 4 × Σ(customer_quarters_active)',
})
_rows.append({
    'Metric': 'CW Expansion (composite)',
    'Target': '(Avg days_before_renewal × Net New ARR) ÷ (Expansion logos × 2,000,000)',
})
print('CS Scorecard Targets')
print('=' * 90)
for r in _rows:
    print(f"  {r['Metric']:<50} {r['Target']}")
print('=' * 90)
pd.DataFrame(_rows).set_index('Metric')

,Target
NRR,1.35
GRR,0.95
LOGO_RETENTION,0.95
STRATEGIC_TIME_TO_ONBOARD,0.00
CORE_PLUS_TIME_TO_ONBOARD,0.00
CORE_TIME_TO_ONBOARD,0.00
SEAT_UTILIZATION,0.80
GONG_MEETINGS_PER_CUSTOMER,4.00
AVG_ACTIONS_MOM,0.75
AVG_BUILDING_SIGNALS,2500.00


## Weights

Each CS owner's overall score is a weighted average of their metric-vs-target attainment. Weights must sum to **100%**.

In [ ]:
WEIGHTS = {
    'NRR':                        0.25,
    'GRR':                        0.25,
    'LOGO_RETENTION':             0.10,
    'STRATEGIC_TIME_TO_ONBOARD':  0.00,
    'CORE_PLUS_TIME_TO_ONBOARD':  0.00,
    'CORE_TIME_TO_ONBOARD':       0.00,
    'SEAT_UTILIZATION':           0.20,
    'GONG_MEETINGS'            : 0.10,
    'AVG_ACTIONS_MOM':            0.00,
    'AVG_BUILDING_SIGNALS':       0.00,
    'CW_EXPANSION':               0.10,
}

_total = round(sum(WEIGHTS.values()), 4)
print('CS Scorecard Weights')
print('=' * 60)
for k, v in WEIGHTS.items():
    marker = ' ' if v > 0 else 'x'
    print(f'  [{marker}] {k:<28} {v:.0%}')
print('-' * 60)
print(f'  {"TOTAL":<32} {_total:.0%}')
print('=' * 60)
if _total != 1.0:
    diff = (_total - 1.0)
    print()
    print(f'!!! ALERT: weights sum to {_total:.2%}, off by {diff:+.2%}. Adjust so total = 100%.')
else:
    print('OK - weights sum to 100%.')

import pandas as pd


## Cell 3 — CS Scorecard aggregations

TBD — aggregate `df_scorecard` (and `df_deals` where relevant) into per-CS-owner scorecards. Definition to be specified by user.

In [6]:
# Per-CS-owner scorecard.
# Aggregate df_scorecard, compute metrics, score = min(success_rate * weight, weight).

import pandas as pd
import numpy as np

# CS owners to exclude from the scorecard (not scored)
EXCLUDED_CS_OWNERS = ['Daniel Chupak', 'Al Sharma', 'Bill Gilleran', 'Nathan Roys', 'Eric Fernandez']

df = df_scorecard[
    df_scorecard['CS_OWNER'].notna()
    & ~df_scorecard['CS_OWNER'].isin(EXCLUDED_CS_OWNERS)
].copy()

# Fallback if the Targets cell hasn't been run in this kernel session
_GONG_PER_CQ = globals().get('GONG_MEETINGS_PER_CUSTOMER_PER_QUARTER', 4)

# Tolerant weight lookup (handles either Gong weight key name across kernel states)
def _w(*keys, default=0.0):
    for k in keys:
        if k in WEIGHTS:
            return WEIGHTS[k]
    return default

# --- Per-customer quarters active (whole calendar months / 3) for Gong target ---
YEAR_START = pd.Timestamp(pd.Timestamp.today().year, 1, 1)
_q_start   = pd.Timestamp(pd.Timestamp.today()).to_period('Q').start_time
WINDOW_END = (_q_start - pd.Timedelta(days=1)).normalize()   # end of last full quarter

_first        = pd.to_datetime(df['FIRST_LICENCE_START'])
_window_start = _first.where(_first > YEAR_START, YEAR_START)
_months = ((WINDOW_END.year - _window_start.dt.year) * 12
           + (WINDOW_END.month - _window_start.dt.month) + 1).clip(lower=0)
df['_QUARTERS_ACTIVE'] = _months / 3.0     # full Jan-Jun window = 6/3 = 2.0

cs = (
    df.groupby('CS_OWNER', dropna=False)
      .agg(
          CS_HIRE_DATE          = ('CS_HIRE_DATE',           'first'),
          CS_RAMP_UP_PCT        = ('CS_RAMP_UP_PCT',         'first'),
          LOGOS                 = ('COMPANY_CRM_ID',         'count'),
          STRATEGIC             = ('IS_STRATEGIC',           'sum'),
          CORE_PLUS             = ('IS_CORE_PLUS',           'sum'),
          CORE                  = ('IS_CORE',                'sum'),
          DIGITAL               = ('IS_DIGITAL',             'sum'),
          ARR_START_OF_YEAR     = ('ARR_START_OF_YEAR',      'sum'),
          ARR_LAST_FULL_QUARTER = ('ARR_LAST_FULL_QUARTER',  'sum'),
          ARR_CHANGE            = ('ARR_CHANGE',             'sum'),
          ARR_RETAINED          = ('ARR_RETAINED',           'sum'),
          RETAINED_LOGOS        = ('IS_RETAINED',            'sum'),
          GONG_CALLS            = ('CS_GONG_CALLS',          'sum'),
          SEAT_UTIL_MEETS       = ('SEAT_UTIL_MEETS_TARGET', 'sum'),
          QUARTERS_ACTIVE       = ('_QUARTERS_ACTIVE',       'sum'),
          ACTIONS_MOM_PCT       = ('ACTIONS_MOM_PCT',        'mean'),   # skips NaN
          BUILDING_SIGNALS      = ('TOTAL_BUILDING_SIGNALS', 'mean'),   # skips NaN
          STRATEGIC_TTO         = ('STRATEGIC_TTO_DAYS',     'mean'),
          CORE_PLUS_TTO         = ('CORE_PLUS_TTO_DAYS',     'mean'),
          CORE_TTO              = ('CORE_TTO_DAYS',          'mean'),
      )
      .reset_index()
)

def score(success_rate, weight):
    return np.minimum(success_rate * weight, weight)

# NRR
cs['NRR']                       = cs['ARR_LAST_FULL_QUARTER'] / cs['ARR_START_OF_YEAR'].replace(0, np.nan)
cs['NRR_TARGET']                = TARGETS['NRR']
cs['NRR_SUCCESS_RATE']          = cs['NRR'] / cs['NRR_TARGET']
cs['NRR_SCORE']                 = score(cs['NRR_SUCCESS_RATE'], WEIGHTS['NRR'])

# GRR
cs['GRR']                       = cs['ARR_RETAINED'] / cs['ARR_START_OF_YEAR'].replace(0, np.nan)
cs['GRR_TARGET']                = TARGETS['GRR']
cs['GRR_SUCCESS_RATE']          = cs['GRR'] / cs['GRR_TARGET']
cs['GRR_SCORE']                 = score(cs['GRR_SUCCESS_RATE'], WEIGHTS['GRR'])

# Logo Retention
cs['LOGO_RETENTION']            = cs['RETAINED_LOGOS'] / cs['LOGOS'].replace(0, np.nan)
cs['LOGO_RETENTION_TARGET']     = TARGETS['LOGO_RETENTION']
cs['LOGO_RETENTION_SUCCESS_RATE'] = cs['LOGO_RETENTION'] / cs['LOGO_RETENTION_TARGET']
cs['LOGO_RETENTION_SCORE']      = score(cs['LOGO_RETENTION_SUCCESS_RATE'], WEIGHTS['LOGO_RETENTION'])

# Seat Utilization (share of customers meeting their tenure-based threshold)
cs['SEAT_UTILIZATION']              = cs['SEAT_UTIL_MEETS'] / cs['LOGOS'].replace(0, np.nan)
cs['SEAT_UTILIZATION_TARGET']       = TARGETS['SEAT_UTILIZATION']
cs['SEAT_UTILIZATION_SUCCESS_RATE'] = cs['SEAT_UTILIZATION'] / cs['SEAT_UTILIZATION_TARGET']
cs['SEAT_UTILIZATION_SCORE']        = score(cs['SEAT_UTILIZATION_SUCCESS_RATE'], WEIGHTS['SEAT_UTILIZATION'])

# Gong Meetings (dynamic target = 4 * quarters_active; symmetric success rate)
cs['GONG_ACTUAL_CALLS'] = cs['GONG_CALLS']
cs['GONG_TARGET_CALLS'] = _GONG_PER_CQ * cs['QUARTERS_ACTIVE']
_ratio = cs['GONG_ACTUAL_CALLS'] / cs['GONG_TARGET_CALLS'].replace(0, np.nan)
cs['GONG_SUCCESS_RATE'] = np.where(_ratio <= 1, _ratio, 1 / _ratio)   # peak at exact target
cs['GONG_SCORE']        = score(cs['GONG_SUCCESS_RATE'], _w('GONG_MEETINGS', 'GONG_MEETINGS_PER_CUSTOMER'))

# Actions MoM (ACTIONS_MOM_PCT is 0-100 -> scale to 0-1)
cs['ACTIONS_MOM']              = cs['ACTIONS_MOM_PCT'] / 100.0
cs['ACTIONS_MOM_TARGET']       = TARGETS['AVG_ACTIONS_MOM']
cs['ACTIONS_MOM_SUCCESS_RATE'] = cs['ACTIONS_MOM'] / cs['ACTIONS_MOM_TARGET']
cs['ACTIONS_MOM_SCORE']        = score(cs['ACTIONS_MOM_SUCCESS_RATE'], WEIGHTS['AVG_ACTIONS_MOM'])  # weight 0

# Avg Building Signals
cs['AVG_BUILDING_SIGNALS']              = cs['BUILDING_SIGNALS']
cs['AVG_BUILDING_SIGNALS_TARGET']       = TARGETS['AVG_BUILDING_SIGNALS']
cs['AVG_BUILDING_SIGNALS_SUCCESS_RATE'] = cs['AVG_BUILDING_SIGNALS'] / cs['AVG_BUILDING_SIGNALS_TARGET']
cs['AVG_BUILDING_SIGNALS_SCORE']        = score(cs['AVG_BUILDING_SIGNALS_SUCCESS_RATE'], WEIGHTS['AVG_BUILDING_SIGNALS'])  # weight 0

# Time to Onboarding (display only; weights are 0 so no score)
cs['STRATEGIC_TTO_DAYS'] = cs['STRATEGIC_TTO']
cs['CORE_PLUS_TTO_DAYS'] = cs['CORE_PLUS_TTO']
cs['CORE_TTO_DAYS']      = cs['CORE_TTO']

# CW Expansion / Renewal (from df_deals; attributed to close-time CS owner)
_exp = (
    df_deals[df_deals['CS_OWNER'].notna()]
      .groupby('CS_OWNER')
      .agg(
          CW_EXPANSION_ARR           = ('NET_NEW_ARR',        'sum'),
          CW_EXPANSION_LOGOS         = ('COMPANY_CRM_ID',     'nunique'),
          CW_AVG_DAYS_BEFORE_RENEWAL = ('DAYS_BEFORE_RENEWAL','mean'),
      )
      .reset_index()
)
cs = cs.merge(_exp, on='CS_OWNER', how='left')
cs['CW_EXPANSION_ARR']   = cs['CW_EXPANSION_ARR'].fillna(0)
cs['CW_EXPANSION_LOGOS'] = cs['CW_EXPANSION_LOGOS'].fillna(0)
# Score = (Avg days_before_renewal * Net New ARR) / (Expansion logos * 20,000)
_denom = (cs['CW_EXPANSION_LOGOS'] * 2_000_000).replace(0, np.nan)
cs['CW_EXPANSION_SCORE'] = (
    cs['CW_AVG_DAYS_BEFORE_RENEWAL'].fillna(0) * cs['CW_EXPANSION_ARR']
) / _denom
cs['CW_EXPANSION_SCORE'] = cs['CW_EXPANSION_SCORE'].fillna(0)   # owners with no expansion -> 0
cs['CW_EXPANSION_WEIGHTED_SCORE'] = np.minimum(
    cs['CW_EXPANSION_SCORE'] * _w('CW_EXPANSION'), _w('CW_EXPANSION'))

# Total weighted score
_score_cols = ['NRR_SCORE','GRR_SCORE','LOGO_RETENTION_SCORE',
               'SEAT_UTILIZATION_SCORE','GONG_SCORE',
               'ACTIONS_MOM_SCORE','AVG_BUILDING_SIGNALS_SCORE',
               'CW_EXPANSION_WEIGHTED_SCORE']
cs['TOTAL_SCORE'] = cs[_score_cols].sum(axis=1)

cs = cs.sort_values('TOTAL_SCORE', ascending=False).reset_index(drop=True)

cs

## How the scorecard works

Each CS owner is scored on several metrics, then the metric scores add up to one **Total score**.

**For every metric:**
1. **Actual** — roll up the owner's customers (or deals): a sum, an average, or a share.
2. **Success rate** = Actual ÷ Target — how close they got to goal.
3. **Score** = Success rate × the metric's weight, **capped at that weight**.

**Total score = sum of all metric scores** (max 100%). Metrics that don't apply score 0.

**Where the data comes from**
- Most metrics use the per-customer table (Cell 1), attributed to the customer's *current* CS owner.
- CW Expansion uses the per-deal table (Cell 2), attributed to the CS owner *at the deal's close date*.

**The metrics**
| Metric | Actual | Target |
| --- | --- | --- |
| NRR | Σ end-of-Q ARR ÷ Σ start-of-year ARR | 135% |
| GRR | same but each customer capped at their starting ARR (expansion can't mask churn) | 95% |
| Logo Retention | share of customers still active at quarter end | 95% |
| Seat Utilization | share of customers meeting their own usage bar (30/80/100% by tenure; unlimited always passes) | 80% |
| Gong Meetings | Σ calls vs 4 per customer per quarter (target grows with portfolio & customer tenure) | dynamic |
| Actions MoM | avg share of months where a customer's self-service runs didn't drop | 75% (weight 0) |
| Avg Building Signals | avg building signals per customer | 2,500 (weight 0) |
| Time to Onboarding | avg days to reach Evolution, per tier | shown only (weight 0) |
| CW Expansion | (Avg days_before_renewal × Net New ARR) ÷ (Expansion logos × 2,000,000) | score used directly |

**Good to know (the nuances)**
- **The cap means beating a target earns nothing extra** — you can reach a metric's weight but not exceed it. Wherever actual comfortably clears target, that metric effectively becomes pass/fail.
- **Gong is scored symmetrically:** hitting the target exactly is best; both under-calling and over-calling lose points equally.
- **Two owner attributions coexist:** every metric except CW credits the *current* owner; CW credits the *close-time* owner — so an owner can score on a deal they've since handed off, or miss one that closed before they arrived.
- **NULLs:** Actions MoM and Building Signals averages skip customers with no value; Seat Utilization counts a missing customer as *not passing*.
- **Weights must total 100%** (checked in the Weights cell). Metrics at 0% weight still compute for visibility but add nothing to the total.

## CS owners by total score

In [ ]:
import matplotlib.pyplot as plt

_c = cs.sort_values('TOTAL_SCORE', ascending=True)   # ascending so largest lands on top in barh
fig, ax = plt.subplots(figsize=(9, max(4, 0.4 * len(_c))))
ax.barh(_c['CS_OWNER'], _c['TOTAL_SCORE'])
ax.set_xlabel('Total score')
ax.set_xlim(0, 1)
ax.set_title('CS Owner Total Score')
for y, v in enumerate(_c['TOTAL_SCORE']):
    ax.text(v + 0.005, y, f'{v:.0%}', va='center', fontsize=8)
plt.tight_layout()
plt.show()

## Success rate per metric

In [ ]:
# Success rate per metric (actual / target). CW has no target-based rate; its raw score is shown separately.
_success_cols = ['NRR_SUCCESS_RATE','GRR_SUCCESS_RATE','LOGO_RETENTION_SUCCESS_RATE',
                 'SEAT_UTILIZATION_SUCCESS_RATE','GONG_SUCCESS_RATE',
                 'ACTIONS_MOM_SUCCESS_RATE','AVG_BUILDING_SIGNALS_SUCCESS_RATE']
cs.set_index('CS_OWNER')[_success_cols].sort_index()

## Actual value per metric

In [ ]:
# Actual value per metric
_actual_cols = ['NRR','GRR','LOGO_RETENTION','SEAT_UTILIZATION',
                'GONG_ACTUAL_CALLS','GONG_TARGET_CALLS','ACTIONS_MOM','AVG_BUILDING_SIGNALS',
                'CW_EXPANSION_LOGOS','CW_EXPANSION_ARR','CW_AVG_DAYS_BEFORE_RENEWAL',
                'STRATEGIC_TTO_DAYS','CORE_PLUS_TTO_DAYS','CORE_TTO_DAYS']
cs.set_index('CS_OWNER')[_actual_cols].sort_index()

## Weighted score per metric

In [ ]:
# Weighted score per metric (contributes to TOTAL_SCORE)
_score_view = ['NRR_SCORE','GRR_SCORE','LOGO_RETENTION_SCORE',
               'SEAT_UTILIZATION_SCORE','GONG_SCORE',
               'ACTIONS_MOM_SCORE','AVG_BUILDING_SIGNALS_SCORE',
               'CW_EXPANSION_WEIGHTED_SCORE','TOTAL_SCORE']
cs.set_index('CS_OWNER')[_score_view].sort_values('TOTAL_SCORE', ascending=False)